In [6]:
%pip install -U langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19436\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
loader = PyPDFLoader("documents/CPA7-LD-4040.pdf")
pages = loader.load()
print(f"Number of pages loaded: {len(pages)}")
print(pages[0].page_content)

Number of pages loaded: 1
Hydranautics Corporate office 
401 Jones Road, Oceanside, CA 92058, USA   Toll Free: +1-800-CPA-PURE   Tel: +1-760-901-2500   Fax: +1-760-901-2578   
Web: www.membranes.com   Email: hy-info@nitto.com 
 
CPA7-LD-4040 
Specified Performance* ______________________________ 
 
Permeate Flow: 2,300 gpd (8.71 m 3/d) 
Salt Rejection: 99.8% (99.6% minimum)         
 
Test Conditions:       1500 ppm NaCl solution 
 225 psig (1.55 MPa) Applied Pressure 
 77 F (25 C) Operating Temperature 
 15% Permeate Recovery 
 6.5 - 7.0 pH Range 
 
*The Specified Performance is based on data taken after a  minimum of 10 minutes of operation.  Actual testing of elements ma y be done at 
conditions which vary from these exact values; in which case, the performance is normalized back to these standard conditions. Permeate flow 
for individual elements may vary + 25 or - 15 percent from the value specified. 
 
General Product Description** ______________________________  
 
Configurati

In [5]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"Number of chunks created: {len(chunks)}")
print("\n--- Chunk 1 ---")
print(chunks[0].page_content)
print("\n--- Chunk 2 ---")
print(chunks[1].page_content)

Number of chunks created: 8

--- Chunk 1 ---
Hydranautics Corporate office 
401 Jones Road, Oceanside, CA 92058, USA   Toll Free: +1-800-CPA-PURE   Tel: +1-760-901-2500   Fax: +1-760-901-2578   
Web: www.membranes.com   Email: hy-info@nitto.com 
 
CPA7-LD-4040 
Specified Performance* ______________________________ 
 
Permeate Flow: 2,300 gpd (8.71 m 3/d) 
Salt Rejection: 99.8% (99.6% minimum)         
 
Test Conditions:       1500 ppm NaCl solution 
 225 psig (1.55 MPa) Applied Pressure 
 77 F (25 C) Operating Temperature

--- Chunk 2 ---
77 F (25 C) Operating Temperature 
 15% Permeate Recovery 
 6.5 - 7.0 pH Range 
 
*The Specified Performance is based on data taken after a  minimum of 10 minutes of operation.  Actual testing of elements ma y be done at 
conditions which vary from these exact values; in which case, the performance is normalized back to these standard conditions. Permeate flow 
for individual elements may vary + 25 or - 15 percent from the value specified.


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
print(api_key is not None)

True


In [15]:
#to search for the correct embedding model name
import google.generativeai as genai

for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [16]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key= os.getenv("GEMINI_API_KEY"))

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector store built successfully")


Vector store built successfully


In [17]:
results = vectorstore.similarity_search("what is the maximum pressure drop", k=2)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)
    print()

--- Result 1 ---
Maximum Applied Pressure: 600 psig (4.14 MPa) 
Maximum Chlorine Concentration: < 0.1 ppm 
Maximum Operating Temperature: 113 F (45 C) 
pH Range, Continuous (Cleaning): 2-11 (1-13) 
Maximum Feedwater Turbidity: 1.0 NTU 
Maximum Feedwater SDI (15 mins): 5.0  
Maximum Feed Flow:  16 gpm (3.6 m3/h) 
Minimum Brine Flow:  3 gpm (0.7 m 3/h) 
Maximum Pressure Drop for Each Element: 15 psi (0.10 Mp a)

--- Result 2 ---
^ The limitations shown here are for general use.  For specified pro jects, operation at more conservative values may ensure the best performan ce and longest life of the membrane.   See 
Hydranautics Technical Bulletins for more details. 
 
Disclaimer: The information and data are presented in good faith and in li eu of all warranties. All express or implied warranties, including the warran ties of merchantability and fitness for a



In [20]:
import google.generativeai as genai

def ask_document(question, k=2):
    # Step 1 — retrieve relevant chunks
    results = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in results])

    # Step 2 — build the prompt
    prompt = f"""
You are an expert on the CPA7-LD-4040 RO membrane datasheet.
Answer the operator's question using ONLY the context below.
If the answer is not in the context, say so clearly.

CONTEXT:
{context}

QUESTION:
{question}
"""

    # Step 3 — send to Gemini
    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)
    return response.text

# Test it
answer = ask_document("What does it mean if pressure drop across an element goes above 15 psi?")
print(answer)

The provided context states the "Maximum Pressure Drop for Each Element: 15 psi (0.10 MPa)". However, it does not explain what it means if the pressure drop across an element goes above 15 psi.
